# DGL DGCNN Classification with topologic_fast

This notebook demonstrates Deep Graph Convolutional Neural Network (DGCNN) for graph classification
using **topologic_fast** to generate graph structures from topological models.

## Overview

DGCNN (Deep Graph CNN) is designed for graph-level classification tasks. It uses:
1. Graph convolutions to learn node representations
2. SortPooling to create a fixed-size representation regardless of graph size
3. 1D convolutions and dense layers for classification

## Prerequisites

```bash
pip install topologic_fast dgl torch pandas plotly scikit-learn
```

## Import Libraries

In [ ]:
# Core imports
import topologic_fast as tf
import numpy as np
import pandas as pd
from pathlib import Path

# DGL and PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GraphConv, SAGEConv, SortPooling
from dgl.dataloading import GraphDataLoader

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(f"topologic_fast loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"DGL version: {dgl.__version__}")

## Create Building Topologies with topologic_fast

We'll create different types of building structures and extract their graph representations.

In [ ]:
def create_multi_cell_building(config):
    """
    Create a building with multiple cells based on configuration.
    
    Args:
        config: dict with 'type', 'params' keys
        
    Returns:
        CellComplex topology
    """
    cells = []
    building_type = config['type']
    
    if building_type == 'tower':
        # Vertical stack of cells
        width = config.get('width', 2)
        length = config.get('length', 2)
        floors = config.get('floors', 5)
        floor_height = config.get('floor_height', 3.0)
        
        for f in range(floors):
            z = f * floor_height
            cell = tf.Cell.Box(0, 0, z, width, length, floor_height)
            cells.append(cell)
            
    elif building_type == 'courtyard':
        # U-shaped building around a courtyard
        size = config.get('size', 4)
        floors = config.get('floors', 3)
        floor_height = 3.0
        
        for f in range(floors):
            z = f * floor_height
            # Left wing
            cells.append(tf.Cell.Box(0, 0, z, 1, size, floor_height))
            # Right wing
            cells.append(tf.Cell.Box(size-1, 0, z, 1, size, floor_height))
            # Back
            cells.append(tf.Cell.Box(1, size-1, z, size-2, 1, floor_height))
            
    elif building_type == 'grid':
        # Regular grid of cells
        nx = config.get('nx', 3)
        ny = config.get('ny', 3)
        nz = config.get('nz', 2)
        cell_size = config.get('cell_size', 1.0)
        
        for i in range(nx):
            for j in range(ny):
                for k in range(nz):
                    x = i * cell_size
                    y = j * cell_size
                    z = k * cell_size * 3  # 3m floor height
                    cell = tf.Cell.Box(x, y, z, cell_size, cell_size, cell_size * 3)
                    cells.append(cell)
                    
    elif building_type == 'stepped':
        # Stepped pyramid
        base_size = config.get('base_size', 5)
        levels = config.get('levels', 4)
        floor_height = 3.0
        
        for level in range(levels):
            size = base_size - level
            z = level * floor_height
            offset = level * 0.5
            for i in range(size):
                for j in range(size):
                    cell = tf.Cell.Box(offset + i, offset + j, z, 1, 1, floor_height)
                    cells.append(cell)
                    
    elif building_type == 'linear':
        # Linear arrangement
        length = config.get('length', 6)
        floors = config.get('floors', 2)
        floor_height = 3.0
        
        for i in range(length):
            for f in range(floors):
                z = f * floor_height
                cell = tf.Cell.Box(i, 0, z, 1, 2, floor_height)
                cells.append(cell)
    
    if len(cells) > 0:
        return tf.CellComplex.ByCells(cells)
    return None

# Create sample topologies
print("Creating sample building topologies...")

tower = create_multi_cell_building({'type': 'tower', 'width': 2, 'length': 2, 'floors': 4})
print(f"Tower: {tower}")

courtyard = create_multi_cell_building({'type': 'courtyard', 'size': 4, 'floors': 2})
print(f"Courtyard: {courtyard}")

grid_building = create_multi_cell_building({'type': 'grid', 'nx': 3, 'ny': 3, 'nz': 2})
print(f"Grid: {grid_building}")

## Extract Graph Features from Topology

Convert topological structures to graph representations suitable for DGCNN.

In [ ]:
def topology_to_graph_data(topology):
    """
    Extract comprehensive graph data from a topology.
    
    Returns:
        dict with 'adj_list', 'node_features', 'edge_index', etc.
    """
    # Create dual graph from topology
    graph = tf.Graph.ByTopology(topology, direct=True, tolerance=0.001)
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    adj_list = graph.AdjacencyList()
    adj_matrix = graph.AdjacencyMatrix()
    
    num_nodes = graph.Order()
    num_edges = graph.Size()
    
    # Node features: coordinates + degree
    node_features = []
    for i, v in enumerate(vertices):
        x, y, z = v.Coordinates()
        degree = len(adj_list[i]) if i < len(adj_list) else 0
        node_features.append([x, y, z, degree])
    
    # Build edge index
    src_nodes = []
    dst_nodes = []
    for i, neighbors in enumerate(adj_list):
        for j in neighbors:
            src_nodes.append(i)
            dst_nodes.append(j)
    
    return {
        'num_nodes': num_nodes,
        'num_edges': num_edges,
        'node_features': np.array(node_features) if node_features else np.zeros((0, 4)),
        'src_nodes': src_nodes,
        'dst_nodes': dst_nodes,
        'adj_matrix': np.array(adj_matrix) if adj_matrix else np.zeros((0, 0)),
        'density': graph.Density(),
        'diameter': graph.Diameter() if num_nodes > 0 else 0
    }

# Test extraction
if tower:
    tower_data = topology_to_graph_data(tower)
    print(f"Tower graph: {tower_data['num_nodes']} nodes, {tower_data['num_edges']} edges")
    print(f"  Density: {tower_data['density']:.4f}")
    print(f"  Diameter: {tower_data['diameter']}")

## Define DGCNN Model

Deep Graph Convolutional Neural Network with SortPooling for graph classification.

In [ ]:
class DGCNN(nn.Module):
    """
    Deep Graph Convolutional Neural Network for graph classification.
    
    Architecture:
    1. Multiple graph convolution layers
    2. SortPooling to create fixed-size representation
    3. 1D convolution layers
    4. Dense layers for classification
    """
    def __init__(self, in_feats, hidden_channels, num_classes, k=30, num_layers=4):
        super(DGCNN, self).__init__()
        
        self.k = k  # Number of nodes to keep in SortPooling
        self.num_layers = num_layers
        
        # Graph convolution layers
        self.convs = nn.ModuleList()
        self.convs.append(GraphConv(in_feats, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GraphConv(hidden_channels, hidden_channels))
        
        # SortPooling layer
        # NOTE: DGL's SortPooling implementation
        self.sort_pool_k = k
        total_latent = hidden_channels * num_layers
        
        # 1D convolution layers
        self.conv1d_1 = nn.Conv1d(1, 16, kernel_size=total_latent, stride=total_latent)
        self.conv1d_2 = nn.Conv1d(16, 32, kernel_size=5, stride=1)
        
        # Calculate feature size after convolutions
        # After conv1d_1: k features
        # After conv1d_2: k - 4 features
        self.pool = nn.MaxPool1d(2, 2)
        
        # Dense layers - adjust size based on k
        dense_input = 32 * max(1, (k - 4) // 2)
        self.dense1 = nn.Linear(dense_input, 128)
        self.dense2 = nn.Linear(128, num_classes)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, g, features):
        # Store outputs from each layer
        hidden_reps = []
        h = features
        
        for conv in self.convs:
            h = F.relu(conv(g, h))
            hidden_reps.append(h)
        
        # Concatenate all hidden representations
        h = torch.cat(hidden_reps, dim=1)
        
        # Custom sort pooling
        g.ndata['h'] = h
        
        # Get graph-level representation via mean pooling
        # (simplified version - full DGCNN uses SortPooling)
        hg = dgl.mean_nodes(g, 'h')
        
        # Classification
        hg = self.dropout(hg)
        hg = F.relu(self.dense1(hg))
        hg = self.dropout(hg)
        out = self.dense2(hg)
        
        return out


class SimplifiedDGCNN(nn.Module):
    """
    Simplified DGCNN using global pooling instead of SortPooling.
    Better suited for varying graph sizes.
    """
    def __init__(self, in_feats, hidden_channels, num_classes, num_layers=4):
        super(SimplifiedDGCNN, self).__init__()
        
        # Graph convolution layers with skip connections
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(SAGEConv(in_feats, hidden_channels, 'mean'))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, 'mean'))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Readout MLP
        self.readout = nn.Sequential(
            nn.Linear(hidden_channels * num_layers, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_channels, num_classes)
        )
        
    def forward(self, g, features):
        hidden_reps = []
        h = features
        
        for conv, bn in zip(self.convs, self.bns):
            h = conv(g, h)
            h = bn(h)
            h = F.relu(h)
            hidden_reps.append(h)
        
        # Concatenate representations from all layers
        h_concat = torch.cat(hidden_reps, dim=1)
        g.ndata['h'] = h_concat
        
        # Global mean pooling
        hg = dgl.mean_nodes(g, 'h')
        
        return self.readout(hg)

print("DGCNN models defined.")

## Generate Training Dataset

In [ ]:
def create_dgl_graph_from_topology(topology, label):
    """
    Create a DGL graph from a topologic_fast topology.
    """
    graph_data = topology_to_graph_data(topology)
    
    num_nodes = graph_data['num_nodes']
    if num_nodes == 0:
        return None
    
    # Create DGL graph
    if len(graph_data['src_nodes']) > 0:
        g = dgl.graph((graph_data['src_nodes'], graph_data['dst_nodes']), num_nodes=num_nodes)
    else:
        # Self-loops for isolated nodes
        g = dgl.graph((list(range(num_nodes)), list(range(num_nodes))), num_nodes=num_nodes)
    
    # Add node features
    g.ndata['feat'] = torch.tensor(graph_data['node_features'], dtype=torch.float32)
    g.label = label
    
    return g


def generate_dgcnn_dataset(samples_per_class=20):
    """
    Generate dataset with different building configurations.
    
    Classes:
    0: Tower buildings
    1: Courtyard buildings
    2: Grid buildings
    3: Stepped buildings
    4: Linear buildings
    """
    graphs = []
    labels = []
    
    np.random.seed(42)
    
    configs = [
        # Tower variations
        [{'type': 'tower', 'width': np.random.randint(1, 4), 'length': np.random.randint(1, 4), 
          'floors': np.random.randint(3, 8)} for _ in range(samples_per_class)],
        
        # Courtyard variations
        [{'type': 'courtyard', 'size': np.random.randint(3, 6), 
          'floors': np.random.randint(2, 5)} for _ in range(samples_per_class)],
        
        # Grid variations
        [{'type': 'grid', 'nx': np.random.randint(2, 4), 'ny': np.random.randint(2, 4),
          'nz': np.random.randint(1, 3)} for _ in range(samples_per_class)],
        
        # Stepped variations
        [{'type': 'stepped', 'base_size': np.random.randint(3, 6),
          'levels': np.random.randint(2, 5)} for _ in range(samples_per_class)],
        
        # Linear variations
        [{'type': 'linear', 'length': np.random.randint(4, 8),
          'floors': np.random.randint(1, 4)} for _ in range(samples_per_class)]
    ]
    
    for label, class_configs in enumerate(configs):
        print(f"Generating class {label}...")
        for config in class_configs:
            try:
                topology = create_multi_cell_building(config)
                if topology is not None:
                    g = create_dgl_graph_from_topology(topology, label)
                    if g is not None:
                        graphs.append(g)
                        labels.append(label)
            except Exception as e:
                pass  # Skip failed samples
    
    return graphs, labels

# Generate dataset
print("Generating DGCNN dataset...")
graphs, labels = generate_dgcnn_dataset(samples_per_class=15)
print(f"\nGenerated {len(graphs)} graphs")
print(f"Class distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")

## Train DGCNN Model

In [ ]:
def collate_fn(batch):
    """Collate graphs and labels."""
    graphs = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    return dgl.batch(graphs), labels

if len(graphs) > 10:
    # Prepare data
    dataset = list(zip(graphs, labels))
    train_data, temp_data = train_test_split(dataset, test_size=0.2, random_state=42, stratify=labels)
    val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)
    
    print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
    
    # Data loaders
    train_loader = GraphDataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = GraphDataLoader(val_data, batch_size=8, shuffle=False, collate_fn=collate_fn)
    test_loader = GraphDataLoader(test_data, batch_size=8, shuffle=False, collate_fn=collate_fn)
    
    # Initialize model
    in_feats = 4  # x, y, z, degree
    hidden_channels = 64
    num_classes = 5
    
    model = SimplifiedDGCNN(in_feats, hidden_channels, num_classes, num_layers=4)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    
    # Training
    num_epochs = 100
    train_losses = []
    train_accs = []
    val_accs = []
    
    print("\nTraining DGCNN...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batched_graph, batch_labels in train_loader:
            features = batched_graph.ndata['feat']
            logits = model(batched_graph, features)
            loss = criterion(logits, batch_labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
        
        scheduler.step()
        
        avg_loss = total_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(avg_loss)
        train_accs.append(train_acc)
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batched_graph, batch_labels in val_loader:
                features = batched_graph.ndata['feat']
                logits = model(batched_graph, features)
                _, predicted = torch.max(logits, 1)
                val_total += batch_labels.size(0)
                val_correct += (predicted == batch_labels).sum().item()
        
        val_acc = val_correct / val_total if val_total > 0 else 0
        val_accs.append(val_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    
    print("Training complete!")
else:
    print("Not enough data for training.")

## Visualize Training Progress

In [ ]:
if 'train_losses' in dir() and len(train_losses) > 0:
    epochs = list(range(1, len(train_losses) + 1))
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Training Loss', 'Accuracy'))
    
    # Loss curve
    fig.add_trace(
        go.Scatter(x=epochs, y=train_losses, mode='lines', name='Train Loss',
                   line=dict(color='blue')),
        row=1, col=1
    )
    
    # Accuracy curves
    fig.add_trace(
        go.Scatter(x=epochs, y=train_accs, mode='lines', name='Train Accuracy',
                   line=dict(color='blue')),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(x=epochs, y=val_accs, mode='lines', name='Val Accuracy',
                   line=dict(color='orange')),
        row=1, col=2
    )
    
    fig.update_layout(
        title='DGCNN Training Progress',
        height=400, width=900
    )
    fig.update_xaxes(title_text='Epoch')
    fig.update_yaxes(title_text='Loss', row=1, col=1)
    fig.update_yaxes(title_text='Accuracy', row=1, col=2)
    
    fig.show()

## Test Model and Plot Confusion Matrix

In [ ]:
if 'model' in dir() and 'test_loader' in dir():
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batched_graph, batch_labels in test_loader:
            features = batched_graph.ndata['feat']
            logits = model(batched_graph, features)
            _, predicted = torch.max(logits, 1)
            all_preds.extend(predicted.tolist())
            all_labels.extend(batch_labels.tolist())
    
    test_acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    class_names = ['Tower', 'Courtyard', 'Grid', 'Stepped', 'Linear']
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=class_names,
        y=class_names,
        colorscale='Blues',
        text=cm,
        texttemplate='%{text}',
        textfont={'size': 14}
    ))
    
    fig.update_layout(
        title='DGCNN Confusion Matrix',
        xaxis_title='Predicted',
        yaxis_title='Actual',
        width=600, height=500
    )
    fig.show()

## Visualize Sample Building Graphs

In [ ]:
def plot_building_graph_3d(topology, title):
    """
    Create 3D visualization of building graph.
    """
    graph = tf.Graph.ByTopology(topology, direct=True, tolerance=0.001)
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Node positions
    x_nodes = [v.X() for v in vertices]
    y_nodes = [v.Y() for v in vertices]
    z_nodes = [v.Z() for v in vertices]
    
    # Edge lines
    x_edges, y_edges, z_edges = [], [], []
    for edge in edges:
        s, e = edge.StartVertex(), edge.EndVertex()
        x_edges.extend([s.X(), e.X(), None])
        y_edges.extend([s.Y(), e.Y(), None])
        z_edges.extend([s.Z(), e.Z(), None])
    
    fig = go.Figure()
    
    # Edges
    fig.add_trace(go.Scatter3d(
        x=x_edges, y=y_edges, z=z_edges,
        mode='lines',
        line=dict(color='lightgray', width=2),
        name='Adjacency'
    ))
    
    # Nodes colored by height
    fig.add_trace(go.Scatter3d(
        x=x_nodes, y=y_nodes, z=z_nodes,
        mode='markers',
        marker=dict(size=6, color=z_nodes, colorscale='Plasma', opacity=0.9),
        name='Cells'
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        width=500, height=450,
        margin=dict(l=0, r=0, t=40, b=0)
    )
    
    return fig

# Show different building types
building_configs = [
    ({'type': 'tower', 'width': 2, 'length': 2, 'floors': 5}, 'Tower'),
    ({'type': 'courtyard', 'size': 4, 'floors': 3}, 'Courtyard'),
    ({'type': 'stepped', 'base_size': 4, 'levels': 3}, 'Stepped')
]

for config, name in building_configs:
    try:
        topo = create_multi_cell_building(config)
        if topo:
            fig = plot_building_graph_3d(topo, f"{name} Building Graph")
            fig.show()
    except Exception as e:
        print(f"Could not visualize {name}: {e}")

## Notes on DGCNN with topologic_fast

### Key Features Demonstrated

1. **Graph Extraction**: `tf.Graph.ByTopology()` creates dual graphs from topological structures
2. **Feature Extraction**: Node coordinates and graph properties (degree, density, diameter)
3. **DGCNN Architecture**: Graph convolutions with global pooling for classification

### Features Not Yet Implemented in topologic_fast

The following topologicpy DGL features require manual implementation:

```python
# NOT YET AVAILABLE - Use manual conversion shown above
# DGL.GraphsByDGCNNPath()  
# DGL.DatasetByGraphs()
# DGL.Hyperparameters()
# DGL.Model()
# DGL.ModelTrain()
```

### Performance Benefits

- Graph extraction from large CellComplexes is significantly faster
- Memory-efficient handling of complex topologies
- Thread-safe operations for parallel dataset generation

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")